### Structured output 
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforing structured output.

### Pydantic
pydantic models provides the richest feature set with field validation , descriptions and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E5785C9160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E5785C9E80>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from pydantic import BaseModel , Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The movies rating out of 10")
    

In [4]:
model_with_structured = model.with_structured_output(Movie)
model_with_structured

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E5785C9160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E5785C9E80>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description':

In [5]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='**Inception (2010)**\n\nInception is a mind-bending science fiction action film written, co-produced, and directed by Christopher Nolan. The movie explores the concept of shared dreaming, where a team of thieves navigates the subconscious mind to plant an idea instead of stealing one.\n\n**Plot**\n\nThe film follows Cobb (Leonardo DiCaprio), a skilled extractor who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" – planting an idea in someone\'s mind instead of stealing one.\n\nSaito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name, which is wanted by the authorities, and allow him to return to the United States to see his children.\n\nCobb assembles a team of experts, including:\n\n1. Arthur (Joseph Gordon-Levitt), a p

In [6]:
response = model_with_structured.invoke("Provide details about the moview Inception")
response 

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

In [7]:
## Message out alongside parsed structred 
from pydantic import BaseModel , Field

class Movie(BaseModel):
    """A movie with details."""
    title:str = Field(... , description="The title of the movie"),
    year:int = Field(..., description="The year the movie was released"),
    director:str= Field(...,description="The director of the movie"),
    rating:float = Field(...,description="The movie's rating out of 10")


model_with_structured = model.with_structured_output(Movie, include_raw=True)
response = model_with_structured.invoke("Provide details about the moview Inception")
response


e:\agentAI\agentAI\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\agentAI\agentAI\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The year the movie was released'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\agentAI\agentAI\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The director of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(m

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dvzfjtaaa', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 260, 'total_tokens': 292, 'completion_time': 0.071829232, 'completion_tokens_details': None, 'prompt_time': 0.013772125, 'prompt_tokens_details': None, 'queue_time': 0.257254494, 'total_time': 0.085601357}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef549-2d82-7c01-bb21-fc314eae8d39-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': 'dvzfjtaaa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 260, 'output_tokens': 32,

In [8]:
##Nested structure 
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genras: list[str]
    budget: float | None = Field(
        None,
        description="Budget in millions USD"
    )

    

model_with_structured = model.with_structured_output(MovieDetails)

response = model_with_structured.invoke("Provide details about the moview Inception")
response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur')], genras=['Action', 'Sci-Fi'], budget=160.0)

## TypeDict
TypeDict provides a simpler alternative using python's built-in typing , ideal when you don't need runtime validation.

In [9]:
from typing_extensions import TypedDict , Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title:Annotated[str , ... ,"The title of the movie"]
    year:Annotated[int , ... , "The year the movie was released"]
    director:Annotated[str , ... , "The director of the movie"]
    rating:Annotated[float , ... , "The movie's rating out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie avengers")

print(response)

{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avengers', 'year': 2012}


In [11]:
class Actor(TypedDict):
    name:str 
    role:str


class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(None , description="Buget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers',
 'year': 2012}

In [12]:
model.profile

{'name': 'Llama 3.3 70B Versatile',
 'release_date': '2024-12-06',
 'last_updated': '2024-12-06',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DataClasses 
A data class is a class typically containimg mainly data , although here are't really amy restrictions.You create if using the @dataclass decorator


In [18]:
import os
os.environ["Gemini_API_Key"] = os.getenv("Gemini_API_Key")

In [21]:
from pydantic import BaseModel , Field
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name:str = Field(description="The name of the person")
    email:str = Field(description="The email address of the person")
    phone:str = Field(description="The phone number of the person")


model = ChatGoogleGenerativeAI(
      model = "gemini-2.5-flash-lite",
      google_api_key=os.getenv("Gemini_API_Key")
)

agent = create_agent(
    model = model,
    response_format=ContactInfo
)
result = agent.invoke({
    "messages":[{"role":"user" , "content":"Extract contact info from: John Doe , john@example.com , (555) 123-4567"}]
})

result
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [22]:
## TypeDict 
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

class ContactInfo(TypedDict):
    name:str
    email:str
    phone:str


model = ChatGoogleGenerativeAI(
      model = "gemini-2.5-flash-lite",
      google_api_key=os.getenv("Gemini_API_Key")
)

agent = create_agent(
    model = model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user" , "content":"Extract contact info from: John Doe , john@example.com , (555) 123-4567"}]
})

result


{'messages': [HumanMessage(content='Extract contact info from: John Doe , john@example.com , (555) 123-4567', additional_kwargs={}, response_metadata={}, id='1f745bd3-1883-450b-8198-635555bed91f'),
  AIMessage(content='{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "(555) 123-4567"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ef567-45d9-7402-ae02-bd3ecc8ec7f1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})],
 'structured_response': {'name': 'John Doe',
  'email': 'john@example.com',
  'phone': '(555) 123-4567'}}

In [23]:
result['structured_response']

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [25]:
## DataClass 
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

@dataclass
class ContactInfo:
    name :str 
    email:str
    phone:str 


model = ChatGoogleGenerativeAI(
      model = "gemini-2.5-flash-lite",
      google_api_key=os.getenv("Gemini_API_Key")
)

agent = create_agent(
    model = model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user" , "content":"Extract contact info from: John Doe , john@example.com , (555) 123-4567"}]
})

result['structured_response']



ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')